# Annotation des verbatims JDMA par LLM (5 dimensions émotionnelles)

Ce notebook annote `data/verbatim_2026_07_31.csv` avec le prompt de `prompt-LLM.rtf` et quatre LLM open-weights,
puis compare leurs annotations (accord inter-modèles, consensus, lien avec les Z-scores de l'analyse factorielle).

**Choix techniques**
- Inférence avec **vLLM** (batching continu) : ~160 000 textes × 4 modèles, impraticable avec `model.generate` de `transformers`.
- **Décodage contraint par schéma JSON** : chaque réponse est un JSON avec exactement les 5 clés et des entiers 0–3.
- **Température 0**, raisonnement (« thinking ») désactivé pour Qwen / Gemma.
- **Sauvegarde par blocs** dans `data/annotations/<RUN_NAME>/<slug>.csv` : si le noyau plante, on relance la cellule et seuls les textes manquants sont traités.

**Prérequis** : GPU NVIDIA (≈ 60 Go de VRAM en bf16 pour les modèles de 22–27B, ou plusieurs GPU via `TENSOR_PARALLEL_SIZE`), et vLLM :
```bash
uv add vllm krippendorff
```

## Imports et configuration

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["HF_HOME"] = os.path.abspath("../cache/")  # même cache que model_trainer.ipynb
os.environ.setdefault("VLLM_WORKER_MULTIPROC_METHOD", "spawn")

import gc
import json
import re
import time
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from sklearn.metrics import cohen_kappa_score

In [ ]:
# ---------------------------------------------------------------- Paramètres
DATA_PATH = Path("../data/verbatim_2026_07_31.csv")
ZSCORES_PATH = Path("../data/Zscores_dimensions.csv")

# Pilote d'abord : N_SAMPLE textes tirés au hasard (mêmes textes pour les 4 modèles).
# Mettre N_SAMPLE = None pour annoter tout le corpus.
N_SAMPLE = 1000
SEED = 1234
RUN_NAME = f"pilot_{N_SAMPLE}" if N_SAMPLE else "full"
OUT_DIR = Path("../data/annotations") / RUN_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)

CHUNK_SIZE = 2000            # textes envoyés à vLLM entre deux sauvegardes
MAX_MODEL_LEN = 8192         # le texte le plus long fait ~8 000 caractères (~2 500 tokens)
MAX_NEW_TOKENS = 96          # le JSON de réponse fait ~40 tokens
TENSOR_PARALLEL_SIZE = 1     # nombre de GPU par modèle
GPU_MEMORY_UTILIZATION = 0.90

DIMENSIONS = ["satisfaction", "contestation", "desillusion", "detresse", "perplexite"]

MODELS = [
    {
        "id": "google/gemma-4-26B-A4B-it",
        "slug": "gemma4_26b_a4b",
        "multimodal": True,
    },
    {
        "id": "mistralai/Mistral-Small-3.2-24B-Instruct-2506",
        "slug": "mistral_small_3_2_24b",
        "multimodal": True,
    },
    {
        "id": "Qwen/Qwen3.6-27B",
        "slug": "qwen3_6_27b",
        "multimodal": True,
    },
    {
        "id": "utter-project/EuroLLM-22B-Instruct-2512",
        "slug": "eurollm_22b_instruct_2512",
        "multimodal": False,
    },
]

# Options vLLM spécifiques à certains modèles
VLLM_EXTRA_KWARGS = {
    # Mistral recommande son propre tokenizer / format de poids avec vLLM
    "mistral_small_3_2_24b": {
        "tokenizer_mode": "mistral",
        "config_format": "mistral",
        "load_format": "mistral",
    },
}

## Prompt d'annotation

Repris de `prompt-LLM.rtf`. Le texte à annoter est ajouté à la suite, entre balises `<texte>`.

In [ ]:
PROMPT_ANNOTATION = """Ce [texte] est la narration d'une expérience vécue avec une administration. Utilise uniquement le texte fourni. Pour chaque texte, attribue indépendamment un score entier de 0 à 3 à chacune des cinq dimensions suivantes.

Satisfaction : appréciation positive d’une expérience dans laquelle l’usager peut avancer, atteindre son objectif ou surmonter une difficulté. Comprend le contentement, la joie, la gratitude, la confiance, l’apaisement et le soulagement. Une appréciation comme « tout s’est bien passé » suffit à en indiquer la présence ; un résultat favorable décrit sans appréciation ne suffit pas.

Contestation : colère, indignation ou sentiment d’injustice face à un traitement jugé inacceptable et imputé à un acteur. L’usager considère généralement qu’une autre conduite était possible ou qu’une correction est due. La cible peut être l’administration ou un tiers impliqué dans l’expérience. Un désaccord ou un recours purement factuel ne suffit pas.

Désillusion : déception et affaiblissement de la confiance dans une résolution ou dans l’utilité des efforts engagés. Comprend le découragement, le sentiment d’abandon, l’impuissance, le désespoir, la résignation et le renoncement lorsqu’ils expriment une perte d’espoir. Une déception explicite suffit à en indiquer la présence, sans exiger un renoncement.

Détresse : inquiétude, anxiété ou angoisse face à une conséquence redoutée dont l’issue reste incertaine. L’usager craint ce qui pourrait lui arriver. Distingue cette menace encore ouverte de la perte d’espoir propre à la désillusion. Une situation grave ou urgente ne suffit pas sans expression d’inquiétude.

Perplexité : étonnement, surprise, confusion ou doute devant une situation inattendue ou difficile à interpréter. L’usager ne comprend pas ce qui se passe, ce qui est demandé ou comment poursuivre. L’incertitude concerne la compréhension, tandis que la détresse concerne les conséquences redoutées. La perplexité peut être positive, négative ou neutre. Une simple demande d’information ne suffit pas.

Applique ce barème à chaque dimension :
0 = non exprimée : aucun indice textuel suffisant.
1 = faible : expression légère ou atténuée.
2 = modérée : expression nette, sans marque de forte intensité.
3 = forte : expression très intense.
Mesure l’intensité exprimée, pas ta certitude ni la gravité objective des faits. « Je suis un peu inquiet » correspond à une détresse de 1 ; « je suis angoissé » à 2 ; « cette angoisse m’empêche de dormir » à 3.

Plusieurs dimensions peuvent coexister, y compris satisfaction et émotions négatives. Tous les scores peuvent être nuls. Ne transforme pas automatiquement un obstacle, un comportement ou une insatisfaction générale en émotion particulière.
Prends en compte toute l’expérience racontée, y compris les émotions passées. Pour chaque dimension, retiens l’intensité la plus élevée clairement exprimée. Une résolution heureuse n’efface pas une inquiétude antérieure. Ne compte pas une émotion niée ou seulement hypothétique.
Réponds uniquement avec un objet JSON valide contenant exactement les clés suivantes et des valeurs entières de 0 à 3 :
{"satisfaction": 0, "contestation": 0, "desillusion": 0, "detresse": 0, "perplexite": 0}
"""

def build_messages(text: str) -> list[dict]:
    return [{
        "role": "user",
        "content": f"{PROMPT_ANNOTATION.strip()}\n\n<texte>\n{text}\n</texte>",
    }]

print(build_messages("Exemple de verbatim.")[0]["content"][-300:])

## Données

In [ ]:
verbatim_df = pd.read_csv(DATA_PATH, sep=";", encoding="utf-8-sig")
verbatim_df = verbatim_df.rename(columns={"ID expérience": "id_experience"})
print("Lignes :", len(verbatim_df))

verbatim_df = verbatim_df.drop_duplicates("id_experience")
print("Après dédoublonnage des ID :", len(verbatim_df))

verbatim_df["full_comment"] = (
    verbatim_df["Titre"].fillna("").str.strip()
    + "\n\n"
    + verbatim_df["Description"].fillna("").str.strip()
).str.strip()
verbatim_df = verbatim_df[verbatim_df["full_comment"].str.len() > 0]

if N_SAMPLE:
    corpus_df = verbatim_df.sample(n=min(N_SAMPLE, len(verbatim_df)), random_state=SEED)
else:
    corpus_df = verbatim_df
corpus_df = corpus_df[["id_experience", "full_comment"]].reset_index(drop=True)
print("Textes à annoter :", len(corpus_df))
corpus_df["full_comment"].str.len().describe()

## Sortie structurée et parsing

In [ ]:
JSON_SCHEMA = {
    "type": "object",
    "properties": {dim: {"type": "integer", "enum": [0, 1, 2, 3]} for dim in DIMENSIONS},
    "required": DIMENSIONS,
    "additionalProperties": False,
}


def parse_scores(raw: str) -> dict:
    """Extrait les 5 scores d'une réponse. Renvoie des NaN si la réponse est inexploitable."""
    empty = {dim: np.nan for dim in DIMENSIONS}
    if not isinstance(raw, str):
        return {**empty, "parse_ok": False}
    raw = re.sub(r"<think>.*?</think>", "", raw, flags=re.S)  # au cas où un modèle « réfléchit »
    match = re.search(r"\{.*?\}", raw, flags=re.S)
    if not match:
        return {**empty, "parse_ok": False}
    try:
        obj = json.loads(match.group(0))
    except json.JSONDecodeError:
        return {**empty, "parse_ok": False}
    scores, ok = {}, True
    for dim in DIMENSIONS:
        val = obj.get(dim)
        if isinstance(val, (int, float)) and not isinstance(val, bool) and int(val) == val and 0 <= val <= 3:
            scores[dim] = int(val)
        else:
            scores[dim], ok = np.nan, False
    return {**scores, "parse_ok": ok and set(obj) == set(DIMENSIONS)}


assert parse_scores('{"satisfaction": 0, "contestation": 2, "desillusion": 1, "detresse": 0, "perplexite": 3}')["parse_ok"]
assert not parse_scores("pas de json")["parse_ok"]
assert not parse_scores('{"satisfaction": 5}')["parse_ok"]

## Fonctions d'inférence (vLLM)

In [ ]:
from vllm import LLM, SamplingParams

# L'API des sorties structurées a changé selon les versions de vLLM
try:
    from vllm.sampling_params import StructuredOutputsParams

    def make_sampling_params() -> SamplingParams:
        return SamplingParams(
            temperature=0.0, max_tokens=MAX_NEW_TOKENS, seed=SEED,
            structured_outputs=StructuredOutputsParams(json=JSON_SCHEMA),
        )
except ImportError:
    from vllm.sampling_params import GuidedDecodingParams

    def make_sampling_params() -> SamplingParams:
        return SamplingParams(
            temperature=0.0, max_tokens=MAX_NEW_TOKENS, seed=SEED,
            guided_decoding=GuidedDecodingParams(json=JSON_SCHEMA),
        )


def load_llm(model: dict) -> LLM:
    kwargs = dict(
        model=model["id"],
        dtype="bfloat16",
        max_model_len=MAX_MODEL_LEN,
        tensor_parallel_size=TENSOR_PARALLEL_SIZE,
        gpu_memory_utilization=GPU_MEMORY_UTILIZATION,
        seed=SEED,
        trust_remote_code=True,
    )
    if model["multimodal"]:
        # Annotation texte seul : on désactive l'encodeur d'images pour économiser la VRAM
        kwargs["limit_mm_per_prompt"] = {"image": 0}
    kwargs.update(VLLM_EXTRA_KWARGS.get(model["slug"], {}))
    return LLM(**kwargs)


def free_llm(llm) -> None:
    """Libère la VRAM avant de charger le modèle suivant."""
    try:
        from vllm.distributed.parallel_state import destroy_model_parallel, destroy_distributed_environment
        destroy_model_parallel()
        destroy_distributed_environment()
    except Exception:
        pass
    del llm
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


def output_path(model: dict) -> Path:
    return OUT_DIR / f"{model['slug']}.csv"


def already_done(model: dict) -> set:
    path = output_path(model)
    if not path.exists():
        return set()
    return set(pd.read_csv(path, usecols=["id_experience"])["id_experience"])


def annotate_with(model: dict) -> None:
    todo = corpus_df[~corpus_df["id_experience"].isin(already_done(model))]
    print(f"\n=== {model['id']} : {len(todo)} textes restants sur {len(corpus_df)}")
    if todo.empty:
        return

    llm = load_llm(model)
    params = make_sampling_params()
    try:
        for start in range(0, len(todo), CHUNK_SIZE):
            chunk = todo.iloc[start:start + CHUNK_SIZE]
            t0 = time.time()
            outputs = llm.chat(
                [build_messages(t) for t in chunk["full_comment"]],
                sampling_params=params,
                use_tqdm=True,
                # Désactive le mode « thinking » (Qwen3.x, Gemma 4) ; ignoré par les autres templates
                chat_template_kwargs={"enable_thinking": False},
            )
            raws = [o.outputs[0].text for o in outputs]
            rows = pd.DataFrame([parse_scores(r) for r in raws])
            rows.insert(0, "id_experience", chunk["id_experience"].to_numpy())
            rows["raw_output"] = raws
            rows["model"] = model["id"]
            path = output_path(model)
            rows.to_csv(path, mode="a", header=not path.exists(), index=False)
            print(f"  {start + len(chunk)}/{len(todo)} — {time.time() - t0:.0f}s — "
                  f"échecs de parsing : {(~rows['parse_ok']).sum()}")
    finally:
        free_llm(llm)

## Annotation

Une cellule par modèle. Si le chargement du modèle suivant échoue faute de VRAM (vLLM ne libère pas toujours tout),
**redémarrer le noyau**, réexécuter les cellules de configuration puis la cellule du modèle concerné : les textes déjà annotés sont sautés.

### Gemma 4 26B-A4B

In [ ]:
annotate_with(MODELS[0])

### Mistral Small 3.2 24B

In [ ]:
annotate_with(MODELS[1])

### Qwen 3.6 27B

In [ ]:
annotate_with(MODELS[2])

### EuroLLM 22B

In [ ]:
annotate_with(MODELS[3])

## Analyse des annotations

Cette partie ne nécessite ni GPU ni vLLM.

In [ ]:
SLUGS = [m["slug"] for m in MODELS]

annotations = {}
for m in MODELS:
    path = output_path(m)
    if path.exists():
        df = pd.read_csv(path).drop_duplicates("id_experience", keep="last")
        annotations[m["slug"]] = df
        print(f"{m['slug']:<28} {len(df):>7} textes, parsing OK : {df['parse_ok'].mean():.1%}")
    else:
        print(f"{m['slug']:<28} pas encore annoté")

available = list(annotations)

In [ ]:
# Format long : une ligne par (texte, modèle, dimension)
long_df = pd.concat(
    [df.assign(slug=slug)[["id_experience", "slug", *DIMENSIONS]] for slug, df in annotations.items()]
).melt(id_vars=["id_experience", "slug"], var_name="dimension", value_name="score")

# Format large : une colonne par (dimension, modèle)
wide_df = long_df.pivot_table(index="id_experience", columns=["dimension", "slug"], values="score")

### Distribution des scores par modèle

In [ ]:
dist = (
    long_df.dropna()
    .groupby(["dimension", "slug"])["score"]
    .value_counts(normalize=True)
    .unstack("score")
    .reindex(columns=[0, 1, 2, 3])
    .fillna(0)
)
fig, axes = plt.subplots(1, len(DIMENSIONS), figsize=(4 * len(DIMENSIONS), 3.5), sharey=True)
for ax, dim in zip(axes, DIMENSIONS):
    dist.loc[dim].reindex(available).plot.barh(stacked=True, ax=ax, colormap="Blues", legend=False, edgecolor="white")
    ax.set_title(dim)
    ax.set_xlim(0, 1)
    ax.set_xlabel("part des textes")
    ax.set_ylabel("")
axes[-1].legend(title="score", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()

display(long_df.groupby(["dimension", "slug"])["score"].mean().unstack().round(2))

### Accord inter-modèles

- **α de Krippendorff ordinal** sur les 4 modèles, par dimension (> 0,8 : fiable ; 0,667–0,8 : acceptable pour des conclusions prudentes).
- **κ de Cohen pondéré quadratiquement** pour chaque paire de modèles.

In [ ]:
def krippendorff_alpha_ordinal(data: np.ndarray, categories=(0, 1, 2, 3)) -> float:
    """data : tableau (annotateurs × items), NaN pour les valeurs manquantes."""
    cats = list(categories)
    k = len(cats)
    idx = {c: i for i, c in enumerate(cats)}
    o = np.zeros((k, k))  # matrice de coïncidences
    for col in data.T:
        vals = [idx[int(v)] for v in col if not np.isnan(v)]
        m = len(vals)
        if m < 2:
            continue
        counts = np.bincount(vals, minlength=k)
        o += (np.outer(counts, counts) - np.diag(counts)) / (m - 1)
    n_c = o.sum(axis=1)
    n = n_c.sum()
    # distance ordinale
    delta = np.zeros((k, k))
    for c in range(k):
        for d in range(k):
            lo, hi = min(c, d), max(c, d)
            delta[c, d] = (n_c[lo:hi + 1].sum() - (n_c[lo] + n_c[hi]) / 2) ** 2
    d_o = (o * delta).sum() / n
    d_e = (np.outer(n_c, n_c) * delta).sum() / (n * (n - 1))
    return 1 - d_o / d_e if d_e > 0 else np.nan


alpha = pd.Series({
    dim: krippendorff_alpha_ordinal(wide_df[dim].reindex(columns=available).to_numpy().T)
    for dim in DIMENSIONS
}, name="alpha_ordinal")
display(alpha.round(3).to_frame())

In [ ]:
if len(available) >= 2:
    fig, axes = plt.subplots(1, len(DIMENSIONS), figsize=(4.2 * len(DIMENSIONS), 4))
    for ax, dim in zip(axes, DIMENSIONS):
        mat = pd.DataFrame(np.eye(len(available)), index=available, columns=available)
        for a, b in combinations(available, 2):
            pair = wide_df[dim][[a, b]].dropna()
            mat.loc[a, b] = mat.loc[b, a] = cohen_kappa_score(pair[a], pair[b], weights="quadratic")
        sns.heatmap(mat, annot=True, fmt=".2f", vmin=0, vmax=1, cmap="Blues", cbar=False, ax=ax,
                    xticklabels=[s.split("_")[0] for s in available],
                    yticklabels=[s.split("_")[0] for s in available] if ax is axes[0] else False)
        ax.set_title(f"{dim}\nα = {alpha[dim]:.2f}")
    plt.suptitle("κ de Cohen pondéré (quadratique) entre modèles", y=1.03)
    plt.tight_layout()
    plt.show()

### Consensus

Score de consensus = **médiane** des modèles (arrondie à l'entier inférieur en cas d'égalité 2–2), avec l'écart max entre modèles
pour repérer les textes litigieux à relire.

In [ ]:
consensus = pd.DataFrame(index=wide_df.index)
for dim in DIMENSIONS:
    scores = wide_df[dim].reindex(columns=available)
    consensus[dim] = np.floor(scores.median(axis=1))
    consensus[f"{dim}_ecart"] = scores.max(axis=1) - scores.min(axis=1)
    consensus[f"{dim}_n_modeles"] = scores.notna().sum(axis=1)
consensus = consensus.reset_index()
consensus.to_csv(OUT_DIR / "consensus.csv", index=False)

display(pd.DataFrame({
    "écart ≥ 2 (part des textes)": [(consensus[f"{d}_ecart"] >= 2).mean() for d in DIMENSIONS],
    "score consensus moyen": [consensus[d].mean() for d in DIMENSIONS],
}, index=DIMENSIONS).round(3))

In [ ]:
# Textes les plus litigieux, à relire manuellement
ecarts = consensus[[f"{d}_ecart" for d in DIMENSIONS]].sum(axis=1)
litigieux = (
    consensus.assign(ecart_total=ecarts)
    .nlargest(10, "ecart_total")
    .merge(corpus_df, on="id_experience")
)
for _, row in litigieux.iterrows():
    print(f"--- {row['id_experience']} (écart total {row['ecart_total']:.0f})")
    print(row["full_comment"][:600])
    display(wide_df.loc[row["id_experience"]].unstack("slug").reindex(index=DIMENSIONS, columns=available))

### Lien avec les Z-scores de l'analyse factorielle

Corrélations de Spearman entre le score de consensus LLM et les Z-scores de chaque dimension.
On attend une diagonale nettement plus forte que le reste de la matrice (validité convergente / discriminante).

In [ ]:
zscores_df = pd.read_csv(ZSCORES_PATH, encoding="utf-8-sig")
check = consensus[["id_experience", *DIMENSIONS]].merge(zscores_df, on="id_experience", suffixes=("_llm", "_z"))
print("Textes appariés :", len(check))

corr = pd.DataFrame(
    [[check[f"{a}_llm"].corr(check[f"{b}_z"], method="spearman") for b in DIMENSIONS] for a in DIMENSIONS],
    index=[f"{d} (LLM)" for d in DIMENSIONS],
    columns=[f"{d} (Z)" for d in DIMENSIONS],
)
fig, ax = plt.subplots(figsize=(7, 5.5))
sns.heatmap(corr, annot=True, fmt=".2f", vmin=-1, vmax=1, cmap="RdBu_r", center=0, ax=ax)
ax.set_title("Spearman : consensus LLM × Z-scores")
plt.tight_layout()
plt.show()